# RT Notebook 25 — D/E Oracle Adversarial Test

**Spec:** `MPF_SIM_D_E_ORACLE_ADVERSARIAL_001`  
**Result family:** `MPF_SIM_D_E_ORACLE_ADVERSARIAL_001_RESULTS_001`  
**Claim ceiling:** `C2_LIMITATION_OR_NEGATIVE_RESULT`

This notebook is self-contained. It creates a frozen specification, generates valid baselines,
applies adversarial mutations, evaluates each row with:

1. a **subject implementation** of the bounded P127/P128-style predicates, and  
2. an **independently expressed declarative oracle**,

then performs shuffled deterministic replay, preserves every disagreement, writes all JSON/JSONL
artifacts, hashes them, and creates a ZIP deliverable.


In [ ]:
from pathlib import Path
import json, hashlib, random, copy, zipfile, platform, sys
from datetime import datetime, timezone

ROOT = Path.cwd()
SPEC_ID = 'MPF_SIM_D_E_ORACLE_ADVERSARIAL_001'
RESULT_ID = 'MPF_SIM_D_E_ORACLE_ADVERSARIAL_001_RESULTS_001'
OUT = ROOT / RESULT_ID
OUT.mkdir(parents=True, exist_ok=True)

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(',', ':'), ensure_ascii=False)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False) + '\n', encoding='utf-8')

def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(canonical(row) + '\n')

print('Output directory:', OUT.resolve())

In [ ]:
SPEC = {'spec_id': 'MPF_SIM_D_E_ORACLE_ADVERSARIAL_001', 'schema_version': '1.0.0', 'status': 'FROZEN_PREEXECUTION_DESIGN', 'research_question': 'Do frozen bounded D/E candidate predicates remain consistent with an independently expressed declarative oracle under adversarial malformed, boundary, transport, and history/witness perturbations?', 'claim_ceiling': 'C2_LIMITATION_OR_NEGATIVE_RESULT', 'independence_controls': ['The oracle is declarative and does not call the subject predicate functions.', 'Adversarial cases are generated from mutation operators rather than copied fixture labels.', 'Expected outcomes are derived from explicit semantic requirements.', 'Replay order is shuffled independently while normalized digests must remain equal.', 'Every disagreement is preserved; no post-hoc deletion or relabeling is allowed.'], 'adversarial_families': ['missing_required_field', 'wrong_type', 'cross_context_transport', 'witness_payload_corruption', 'history_order_corruption', 'threshold_below_boundary', 'threshold_exact_boundary', 'threshold_above_boundary', 'unknown_profile', 'duplicate_or_conflicting_trace', 'combined_faults'], 'seeds': [719, 811, 907], 'contexts': ['C7', 'C8', 'C9'], 'profiles': {'low': 0.2, 'mid': 0.5, 'high': 0.8}, 'replay_passes': 2, 'termination_conditions': ['All declared baseline and adversarial cases complete twice.', 'Any required evaluator component changes after execution begins.', 'Any output cannot be canonically serialized or hashed.', 'Any disagreement is omitted from the falsification report.'], 'allowed_interpretation': 'Finite bounded evidence concerning agreement or disagreement between the frozen subject predicates and the independently expressed oracle over the declared adversarial domain.', 'blocked_interpretations': ['Universal D/E preservation', 'Universal non-collapse', 'Injectivity', 'Reversibility', 'Theorem promotion', 'External physical validity'], 'content_sha256': 'af78788bfa218d3d59ae4d19be9d2c579b4f4909d0f54105ca25a25bf7739540'}
write_json(OUT / 'experiment_spec.json', SPEC)
SPEC

## Frozen semantic requirements

A row is **representable** exactly when:

- its declared type is `SourceRelation`,
- source and target contexts are equal,
- its witness exists and exactly binds the ordered source payload,
- history exists, is ordered, and terminates at the declared target,
- the selected threshold profile is known.

A row is **non-collapsed** exactly when:

- the profile is known,
- distinction is positive,
- distinction is strictly greater than the profile threshold.

The subject and oracle implement these rules through different code structures.


In [ ]:

PROFILES = {"low": 0.20, "mid": 0.50, "high": 0.80}
CONTEXTS = ["C7", "C8", "C9"]
SEEDS = [719, 811, 907]

# Subject under test: branch-oriented candidate implementation.
def subject_representability(row):
    if row.get("relation_type") != "SourceRelation":
        return "REJECT_TYPE"
    if row.get("context") != row.get("target_context"):
        return "REJECT_CONTEXT"
    if row.get("profile") not in PROFILES:
        return "REJECT_PROFILE"
    witness = row.get("witness")
    if not isinstance(witness, dict):
        return "REJECT_WITNESS"
    if witness.get("token") != row.get("source_payload"):
        return "REJECT_WITNESS"
    history = row.get("history")
    if not isinstance(history, list) or len(history) == 0:
        return "REJECT_HISTORY"
    if history != sorted(history, key=lambda x: x.get("step", -1)):
        return "REJECT_HISTORY"
    if history[-1].get("state") != row.get("target"):
        return "REJECT_HISTORY"
    return "REPRESENTABLE"

def subject_noncollapse(row):
    profile = row.get("profile")
    if profile not in PROFILES:
        return "REJECT_PROFILE"
    d = row.get("distinction")
    if not isinstance(d, (int, float)) or isinstance(d, bool) or d <= 0:
        return "REJECT_DISTINCTION"
    if d <= PROFILES[profile]:
        return "REJECT_SUBTHRESHOLD"
    return "NON_COLLAPSED"

# Independent oracle: declarative requirements + precedence table.
REP_REQUIREMENTS = {
    "type_ok": lambda r: r.get("relation_type") == "SourceRelation",
    "context_ok": lambda r: r.get("context") == r.get("target_context"),
    "profile_ok": lambda r: r.get("profile") in PROFILES,
    "witness_shape_ok": lambda r: isinstance(r.get("witness"), dict),
    "witness_binding_ok": lambda r: isinstance(r.get("witness"), dict)
        and r["witness"].get("token") == r.get("source_payload"),
    "history_shape_ok": lambda r: isinstance(r.get("history"), list)
        and len(r["history"]) > 0,
    "history_order_ok": lambda r: isinstance(r.get("history"), list)
        and len(r["history"]) > 0
        and all(isinstance(x, dict) for x in r["history"])
        and [x.get("step") for x in r["history"]] == sorted(x.get("step") for x in r["history"]),
    "history_target_ok": lambda r: isinstance(r.get("history"), list)
        and len(r["history"]) > 0
        and isinstance(r["history"][-1], dict)
        and r["history"][-1].get("state") == r.get("target")
}
REP_PRECEDENCE = [
    ("type_ok", "REJECT_TYPE"),
    ("context_ok", "REJECT_CONTEXT"),
    ("profile_ok", "REJECT_PROFILE"),
    ("witness_shape_ok", "REJECT_WITNESS"),
    ("witness_binding_ok", "REJECT_WITNESS"),
    ("history_shape_ok", "REJECT_HISTORY"),
    ("history_order_ok", "REJECT_HISTORY"),
    ("history_target_ok", "REJECT_HISTORY"),
]

def oracle_representability(row):
    truth = {name: bool(rule(row)) for name, rule in REP_REQUIREMENTS.items()}
    for requirement, rejection in REP_PRECEDENCE:
        if not truth[requirement]:
            return rejection
    return "REPRESENTABLE"

def oracle_noncollapse(row):
    profile_known = row.get("profile") in PROFILES
    if not profile_known:
        return "REJECT_PROFILE"
    d = row.get("distinction")
    numeric_positive = isinstance(d, (int, float)) and not isinstance(d, bool) and d > 0
    if not numeric_positive:
        return "REJECT_DISTINCTION"
    above_threshold = d > PROFILES[row["profile"]]
    return "NON_COLLAPSED" if above_threshold else "REJECT_SUBTHRESHOLD"

print("Frozen subject and declarative oracle loaded.")


In [ ]:

def baseline(seed, context, profile, index):
    rng = random.Random(f"{seed}:{context}:{profile}:{index}")
    payload = [rng.randint(10, 99), rng.randint(100, 999)]
    target = f"T_{context}_{seed}_{index}"
    threshold = PROFILES[profile]
    distinction = round(threshold + 0.10 + rng.random() * 0.09, 6)
    return {
        "row_id": f"B_{seed}_{context}_{profile}_{index}",
        "seed": seed,
        "context": context,
        "target_context": context,
        "relation_type": "SourceRelation",
        "source_payload": payload,
        "witness": {"token": copy.deepcopy(payload), "mode": "enriched"},
        "history": [
            {"step": 0, "state": f"S_{index}"},
            {"step": 1, "state": target}
        ],
        "target": target,
        "profile": profile,
        "distinction": distinction,
        "trace": [{"event": "generated"}, {"event": "bound"}],
        "family": "baseline"
    }

BASELINES = [
    baseline(seed, context, profile, index)
    for seed in SEEDS
    for context in CONTEXTS
    for profile in PROFILES
    for index in range(2)
]
len(BASELINES)


In [ ]:

def mutate(row, family):
    r = copy.deepcopy(row)
    r["parent_id"] = row["row_id"]
    r["family"] = family
    r["row_id"] = f"{row['row_id']}__{family}"

    if family == "missing_required_field":
        r.pop("witness", None)
    elif family == "wrong_type":
        r["relation_type"] = "UnrelatedType"
    elif family == "cross_context_transport":
        r["target_context"] = next(c for c in CONTEXTS if c != r["context"])
    elif family == "witness_payload_corruption":
        r["witness"]["token"] = list(reversed(r["source_payload"]))
    elif family == "history_order_corruption":
        r["history"] = list(reversed(r["history"]))
    elif family == "threshold_below_boundary":
        r["distinction"] = round(PROFILES[r["profile"]] - 0.000001, 6)
    elif family == "threshold_exact_boundary":
        r["distinction"] = PROFILES[r["profile"]]
    elif family == "threshold_above_boundary":
        r["distinction"] = round(PROFILES[r["profile"]] + 0.000001, 6)
    elif family == "unknown_profile":
        r["profile"] = "undeclared"
    elif family == "duplicate_or_conflicting_trace":
        r["trace"].append({"event": "bound"})
        r["trace"].append({"event": "conflict", "target": "OTHER"})
        # Trace is intentionally non-semantic under the frozen predicates.
    elif family == "combined_faults":
        r["relation_type"] = "Wrong"
        r["target_context"] = "CX"
        r["witness"] = None
        r["history"] = []
        r["profile"] = "undeclared"
        r["distinction"] = -1
    else:
        raise ValueError(f"Unknown family: {family}")
    return r

FAMILIES = SPEC["adversarial_families"]
ROWS = []
for b in BASELINES:
    ROWS.append(copy.deepcopy(b))
    ROWS.extend(mutate(b, fam) for fam in FAMILIES)

print("Baselines:", len(BASELINES))
print("Total rows:", len(ROWS))


In [ ]:

def evaluate(row):
    sr = subject_representability(row)
    sn = subject_noncollapse(row)
    orr = oracle_representability(row)
    orn = oracle_noncollapse(row)
    return {
        "row_id": row["row_id"],
        "parent_id": row.get("parent_id"),
        "family": row["family"],
        "subject_representability": sr,
        "oracle_representability": orr,
        "subject_noncollapse": sn,
        "oracle_noncollapse": orn,
        "representability_agreement": sr == orr,
        "noncollapse_agreement": sn == orn,
        "falsification_flag": (sr != orr) or (sn != orn),
        "input_sha256": sha256_bytes(canonical(row).encode("utf-8"))
    }

PASS1 = [evaluate(r) for r in ROWS]

rng = random.Random(2501)
shuffled = copy.deepcopy(ROWS)
rng.shuffle(shuffled)
PASS2_UNORDERED = [evaluate(r) for r in shuffled]

def normalized_digest(results):
    normalized = sorted(results, key=lambda x: x["row_id"])
    return sha256_bytes(canonical(normalized).encode("utf-8"))

digest1 = normalized_digest(PASS1)
digest2 = normalized_digest(PASS2_UNORDERED)
replay_agreement = digest1 == digest2

FLAGS = [x for x in PASS1 if x["falsification_flag"]]
print("Pass 1 digest:", digest1)
print("Pass 2 digest:", digest2)
print("Replay agreement:", replay_agreement)
print("Falsification flags:", len(FLAGS))


In [ ]:

# Additional adversarial invariants independent of direct oracle equality.
by_id = {r["row_id"]: r for r in ROWS}
res_by_id = {r["row_id"]: x for r, x in zip(ROWS, PASS1)}

invariants = []

for b in BASELINES:
    bid = b["row_id"]
    bres = res_by_id[bid]
    invariants.append({
        "invariant": "baseline_admission",
        "row_id": bid,
        "pass": bres["subject_representability"] == "REPRESENTABLE"
            and bres["subject_noncollapse"] == "NON_COLLAPSED"
    })
    for fam in ["threshold_below_boundary", "threshold_exact_boundary"]:
        rid = f"{bid}__{fam}"
        invariants.append({
            "invariant": "threshold_noncollapse_boundary",
            "row_id": rid,
            "pass": res_by_id[rid]["subject_noncollapse"] == "REJECT_SUBTHRESHOLD"
        })
    rid = f"{bid}__threshold_above_boundary"
    invariants.append({
        "invariant": "threshold_strict_above_admission",
        "row_id": rid,
        "pass": res_by_id[rid]["subject_noncollapse"] == "NON_COLLAPSED"
    })
    rid = f"{bid}__duplicate_or_conflicting_trace"
    invariants.append({
        "invariant": "nonsemantic_trace_invariance",
        "row_id": rid,
        "pass": (
            res_by_id[rid]["subject_representability"] == bres["subject_representability"]
            and res_by_id[rid]["subject_noncollapse"] == bres["subject_noncollapse"]
        )
    })

invariant_failures = [x for x in invariants if not x["pass"]]
print("Invariant checks:", len(invariants))
print("Invariant failures:", len(invariant_failures))


In [ ]:

summary = {
    "spec_id": SPEC_ID,
    "result_id": RESULT_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "claim_ceiling": SPEC["claim_ceiling"],
    "counts": {
        "baselines": len(BASELINES),
        "adversarial_families": len(FAMILIES),
        "total_rows_per_pass": len(ROWS),
        "total_evaluations": len(PASS1) + len(PASS2_UNORDERED),
        "oracle_disagreements": len(FLAGS),
        "invariant_checks": len(invariants),
        "invariant_failures": len(invariant_failures)
    },
    "replay": {
        "pass1_normalized_digest": digest1,
        "pass2_normalized_digest": digest2,
        "agreement": replay_agreement
    },
    "outcome": (
        "PASS_BOUNDED_NO_COUNTEREXAMPLE"
        if not FLAGS and not invariant_failures and replay_agreement
        else "COUNTEREXAMPLE_OR_LIMITATION_PRESERVED"
    ),
    "interpretation": SPEC["allowed_interpretation"],
    "blocked_interpretations": SPEC["blocked_interpretations"]
}

falsification_report = {
    "spec_id": SPEC_ID,
    "result_id": RESULT_ID,
    "falsification_count": len(FLAGS),
    "invariant_failure_count": len(invariant_failures),
    "replay_agreement": replay_agreement,
    "oracle_disagreements": FLAGS,
    "invariant_failures": invariant_failures,
    "preservation_rule": "All disagreements and invariant failures are retained without deletion."
}

write_jsonl(OUT / "adversarial_rows.jsonl", ROWS)
write_jsonl(OUT / "evaluations_pass1.jsonl", PASS1)
write_jsonl(OUT / "evaluations_pass2_shuffled.jsonl", PASS2_UNORDERED)
write_jsonl(OUT / "invariant_checks.jsonl", invariants)
write_json(OUT / "summary.json", summary)
write_json(OUT / "falsification_report.json", falsification_report)

summary


In [ ]:

# Hash every output before writing the manifest.
artifact_files = sorted(
    p for p in OUT.iterdir()
    if p.is_file() and p.name != "manifest.json" and p.name != f"{RESULT_ID}.zip"
)

manifest = {
    "spec_id": SPEC_ID,
    "result_id": RESULT_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "runtime": {
        "python": sys.version,
        "platform": platform.platform()
    },
    "artifacts": [
        {
            "name": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_bytes(p.read_bytes())
        }
        for p in artifact_files
    ],
    "spec_content_sha256": SPEC["content_sha256"],
    "replay_agreement": replay_agreement,
    "outcome": summary["outcome"]
}
write_json(OUT / "manifest.json", manifest)

bundle = OUT / f"{RESULT_ID}.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p != bundle:
            z.write(p, arcname=p.name)

print("Bundle:", bundle.resolve())
print("Bundle SHA-256:", sha256_bytes(bundle.read_bytes()))
manifest


## Decision rule

A clean run authorizes only:

> No counterexample was found within the frozen Notebook 25 adversarial domain.

Any oracle disagreement, invariant failure, replay mismatch, serialization failure, or missing artifact
must remain preserved as bounded negative or limitation evidence.
